In [6]:
import pandas as pd 
import numpy as np 
# import matplotlib.pyplot as plt
# from pyfaidx import Fasta
# import regex
from sklearn.model_selection import train_test_split
import os

%matplotlib inline
# pd.options.mode.chained_assignment = None 
np.random.seed = 42

#1 subsample origins with ACS
we need to only consider origin samples including ACS motif, Then subsample non-origins with ACS


Our origin data( 325 oriDb seq<=500) 295 of origins have intersect with ACS motifs in motifs_homer1.bed(with their real range)
we also considerd  3 more: 
in which has ACS motif  located in extended upstream and downstream of sequence which reletively are close to real range of origin instances (<100 bp>):
chr10	683325	683824
chr12	947885	948384
chr11	642303	642802

Finally 298 origins chosed from our 325 oriDb seq<=500 and extend them: ex_Oridb_homer_final.bed : positive samples


#2 subsample non-origins with ACS
We constructed the negative samples in ACS-Neg dataset in a way that their scores distribution be closely matched with the ACS motif score distribution of the positive samples. Specifically, negative instances were subsampled such that no statistically significant difference in ACS motif strength was detected between classes. 

In [ ]:
#loading the positive instances 
bed_columns = ['chr', 'start', 'end']
positive_instances = pd.read_csv("ex_Oridb_homer_final.bed",sep='\t',header=None, names=bed_columns)
#loading all ACS motif matech(using HOMER)in genome which is filltered out from the positive instances(all origins in OriDB)
Filtterd_homer=pd.read_csv("motif_homer1_filterdAlloridb.bed",sep='\t',header=None,  usecols=[0, 1, 2], names=bed_columns)
print(positive_instances)
Filtterd_homer

       chr   start     end
0     chr1   30781   31280
1     chr1   70170   70669
2     chr1  124264  124763
3     chr1  159903  160402
4     chr1  175911  176410
..     ...     ...     ...
293  chr16  776744  777243
294  chr16  818960  819459
295  chr16  842595  843094
296  chr16  880682  881181
297  chr16  932894  933393

[298 rows x 3 columns]


,chr,start,end
0,chr1,2168,2184
1,chr1,3174,3190
2,chr1,4659,4675
3,chr1,5042,5058
4,chr1,5052,5068
...,...,...,...
10007,chr16,939184,939200
10008,chr16,939777,939793
10009,chr16,940487,940503
10010,chr16,940707,940723


Here, I generated negative instances by extending non_replicating ACSs which dont have intersect with positive instances (ex_Oridb_homer_final.bed) and all other origings in oridb.

In [ ]:
# loading all the origins in OriDB
All_origins_oridb= pd.read_csv("/data/yeast_origins_oridbAll.bed",sep='\t',header=None, names=bed_columns)
All_origins_oridb

,chr,start,end
0,chr1,650,1791
1,chr1,6136,7136
2,chr1,7998,8548
3,chr1,9775,10485
4,chr1,16855,17565
...,...,...,...
824,chr16,902206,917206
825,chr16,929725,930435
826,chr16,932976,933223
827,chr16,940923,943157


In [ ]:
# loading all the origins in OriDB
positive_df = positive_instances
positive_df['label']=1


negative_df = pd.DataFrame(columns=['chr', 'start', 'end', 'label'])
negative_windows = []
# for each chromosome block the ranges which are origins
for chr, origins in positive_df.groupby("chr"):
    # collect all blocked indices origins 
    blocked_indices = []
    for index, origin in origins.iterrows():
        blocked_indices.append([*range(origin.start, origin.end)])

    # add blocked indices from All_origins_oridb (for this chromosome)
    extra_origins = All_origins_oridb[All_origins_oridb["chr"] == chr]
    for _, ori in extra_origins.iterrows():
        blocked_indices.append(range(ori.start, ori.end))

    # flatten into a single array
    if blocked_indices:
        blocked_indices = np.hstack([np.fromiter(r, dtype=int) for r in blocked_indices])
    else:
        blocked_indices = np.array([], dtype=int)

    
    
    
    # read from non origin ACS motif matech
    homer_motifs= Filtterd_homer[Filtterd_homer['chr'] == chr]
    num_negatives_yet = len(negative_windows)

    # create as many negative candidate by extending ACS motifs
    for i, motif in homer_motifs.iterrows():
      
      
        len_motif= motif['end']-motif['start']
        delta = 500 - len_motif
        pre =  np.random.randint(0, delta)
        post=delta -pre
        sample_start= motif['start'] - pre
        sample_end= motif['end'] + post
        window_indices = range(sample_start, sample_end)
        if(sample_start>0):
            # check if those indices intersects with the blocked ones
            if len(np.intersect1d(blocked_indices, window_indices)) == 0:
                negative_windows.append([chr, sample_start,sample_end])
                # add new indices to blocked ones so that we have no intersecting windows
                blocked_indices = np.append(blocked_indices, window_indices)
                negative_df.loc[len(negative_df)] = {'chr': motif['chr'], 'start':sample_start, 'end': sample_start+499, 'label':0}
                
              
print(negative_df.head())

print(len(negative_df))
negative_df.to_csv('candidates_ACS_Neg.bed', sep='\t', header=False, index=False)

    chr  start   end  label
0  chr1   1812  2311      0
1  chr1   2808  3307      0
2  chr1   4444  4943      0
3  chr1   5468  5967      0
4  chr1   9184  9683      0
5920


In [ ]:
from pyfaidx import Fasta

sequence_data = Fasta(f"/data/S288C_reference_sequence_R64-3-1_20210421.fsa")

# add the DNA sequence to each region
negative_df["seq"] = negative_df.apply(lambda x: sequence_data[x.chr][x.start-1: x.end].seq, axis=1)

negative_df

,chr,start,end,label,seq
0,chr1,1812,2311,0,TTGCGATAGTGTAGATACCGTCCTTGGATAGAGCACTGGAGATGGC...
1,chr1,2808,3307,0,ATACATATTTAAAAGAGGGTACCGCTAATTTAGCAGGGCAGTATTA...
2,chr1,4444,4943,0,TATCAATCTCAATATTAAACTTTTTAGGCTTTCAGGTTTAATCTTT...
3,chr1,5468,5967,0,TGTAGCACGCGTAACTCCTAAACTTTGTCATAATGGTTGAAATGAA...
4,chr1,9184,9683,0,GGTTTCTTATCTATTTGTGGAGGCACCCGTAGTACTGTGCTTTCGT...
...,...,...,...,...,...
5915,chr9,431693,432192,0,GAGCAGAGAGTGTTATAAGCTTCTGATGAGATGGTAGTCGAAGGAA...
5916,chr9,433490,433989,0,TTAGACAATAAGCTTCTGTACGAGGGTCCAAATCTAAAATTCGGAT...
5917,chr9,434158,434657,0,GTGACTTCACCACCATGTTGACTGGTATTCCAGCTGAACAAGTCAC...
5918,chr9,435728,436227,0,CTTTAGTTTCATTGAGTTCCTTAATGAATTCTAGATATTCCTCCAA...


In [10]:
negative_df.to_csv('candidates_ACS_Neg.tsv', sep="\t", index=False)

In [ ]:
# save negative candidates as fasta file
neg_out = open("candidates_ACS_Neg.fa", "w")

counter = 0
for i, row in negative_df.iterrows():
    seq = row["seq"].upper()
    label = row["label"]

    header = f">seq_{counter}"
    counter += 1
    neg_out.write(f"{header}\n{seq}\n")
    
neg_out.close()

Use candidate_ACS_Neg.fa to run Homer and find all ACS Matech in candidate sequences with their scores, because after extending ACSs to 500 bp there might be more that one matche in each sequences and we need best score. Then use following script to choose the best ACS match for each sequences based on score. 
Install HOMER pachage and 
run following command:
findMotifs.pl candidates_acs_neg.fa fasta candidates_acs_neg_out/ -find acs_motif_matrix.motif > candidates_acs_neg.scan.txt


In [13]:
import pandas as pd

def extract_best(infile, outfile):
    df = pd.read_csv(infile, sep="\t", comment="#")

    print(f"Columns in {infile}:")
    print(df.columns.tolist())

    seq_col = "FASTA ID"
    score_col = "MotifScore"

    df[score_col] = pd.to_numeric(df[score_col], errors="coerce")
    df = df.dropna(subset=[score_col])

    idx = df.groupby(seq_col)[score_col].idxmax()
    best = df.loc[idx, [seq_col, score_col]].copy()
    best.columns = ["sequence_id", "best_score"]

    best.to_csv(outfile, index=False)
    print(f"Saved {outfile}")

extract_best("candidates_acs_neg.scan.txt", "candidates_acs_neg_best.csv")

Columns in candidates_acs_neg2.scan.txt:
['FASTA ID', 'Offset', 'Sequence', 'Motif Name', 'Strand', 'MotifScore']
Saved candidates_acs_neg2.scan_best.csv


Do same for positive instances to fine ASc matches best scores in origins. generate file acs_pos_best.csv 
 
Use Candidate_neg_acs_best.csv and acs_pos_best.csv to select negatives from candidate list , in a way that scores of negatives and positives have close statistical distribution, with same mean.

In [1]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Optional, Tuple

In [2]:
@dataclass
class MatchResult:
    selected_negatives: pd.DataFrame
    unmatched_positive_count: int
    exact_match_count: int
    total_absolute_difference: float
    mean_absolute_difference: float

In [3]:
def load_scores(path: str, score_col: str, id_col: Optional[str] = None) -> pd.DataFrame:
    df = pd.read_csv(path)

    if score_col not in df.columns:
        raise ValueError(f"Missing score column '{score_col}' in {path}. Columns: {df.columns.tolist()}")

    df = df.copy()
    df[score_col] = pd.to_numeric(df[score_col], errors="coerce")
    df = df.dropna(subset=[score_col]).reset_index(drop=True)

    if id_col is not None and id_col not in df.columns:
        raise ValueError(f"Missing id column '{id_col}' in {path}. Columns: {df.columns.tolist()}")

    return df

In [5]:
def summarize_distributions(
    positives: pd.DataFrame,
    negatives: pd.DataFrame,
    pos_score_col: str,
    neg_score_col: str,
) -> dict:
    p = positives[pos_score_col].astype(float).to_numpy()
    n = negatives[neg_score_col].astype(float).to_numpy()

    return {
        "positive_n": int(len(p)),
        "negative_n": int(len(n)),
        "positive_mean": float(np.mean(p)),
        "negative_mean": float(np.mean(n)),
        "positive_median": float(np.median(p)),
        "negative_median": float(np.median(n)),
        "positive_std": float(np.std(p, ddof=1)) if len(p) > 1 else 0.0,
        "negative_std": float(np.std(n, ddof=1)) if len(n) > 1 else 0.0,
    }

In [4]:
def greedy_match_scores(
    positives: pd.DataFrame,
    negatives: pd.DataFrame,
    pos_score_col: str,
    neg_score_col: str,
    random_seed: int = 42,
    shuffle_positives: bool = True,
) -> MatchResult:
    rng = np.random.default_rng(random_seed)

    pos = positives.copy().reset_index(drop=True)
    neg = negatives.copy().reset_index(drop=True)

    if len(neg) < len(pos):
        raise ValueError("Negative candidate pool is smaller than positives.")

    pos["_pos_idx"] = np.arange(len(pos))
    neg["_neg_idx"] = np.arange(len(neg))

    if shuffle_positives:
        pos = pos.sample(frac=1, random_state=random_seed).reset_index(drop=True)

    remaining_neg = neg.copy()
    selected_rows = []
    exact_match_count = 0
    unmatched_positive_count = 0
    total_abs_diff = 0.0

    for _, prow in pos.iterrows():
        pscore = float(prow[pos_score_col])

        exact_hits = remaining_neg[remaining_neg[neg_score_col] == pscore]

        if not exact_hits.empty:
            chosen = exact_hits.sample(n=1, random_state=int(rng.integers(0, 1_000_000))).iloc[0]
            abs_diff = 0.0
            exact_match_count += 1
        else:
            diffs = (remaining_neg[neg_score_col] - pscore).abs()
            if diffs.empty:
                unmatched_positive_count += 1
                continue

            min_diff = diffs.min()
            nearest = remaining_neg[diffs == min_diff]
            chosen = nearest.sample(n=1, random_state=int(rng.integers(0, 1_000_000))).iloc[0]
            abs_diff = float(min_diff)

        total_abs_diff += abs_diff

        row = chosen.to_dict()
        row["_matched_positive_score"] = pscore
        row["_abs_diff"] = abs_diff
        selected_rows.append(row)

        remaining_neg = remaining_neg[remaining_neg["_neg_idx"] != chosen["_neg_idx"]]

    selected = pd.DataFrame(selected_rows)

    if selected.empty:
        raise RuntimeError("No negatives were selected.")

    mean_abs_diff = total_abs_diff / len(selected)

    return MatchResult(
        selected_negatives=selected.reset_index(drop=True),
        unmatched_positive_count=unmatched_positive_count,
        exact_match_count=exact_match_count,
        total_absolute_difference=total_abs_diff,
        mean_absolute_difference=mean_abs_diff,
    )

In [6]:
def run_distribution_check(
    positives: pd.DataFrame,
    negatives: pd.DataFrame,
    pos_score_col: str,
    neg_score_col: str,
) -> Tuple[bool, dict]:
    summary = summarize_distributions(positives, negatives, pos_score_col, neg_score_col)

    mean_diff = abs(summary["positive_mean"] - summary["negative_mean"])
    median_diff = abs(summary["positive_median"] - summary["negative_median"])
    std_diff = abs(summary["positive_std"] - summary["negative_std"])

    passed = (
        summary["positive_n"] == summary["negative_n"]
        and mean_diff < 1e-6
        and median_diff < 1e-6
        and std_diff < 1e-6
    )

    stats = {
        **summary,
        "mean_diff": mean_diff,
        "median_diff": median_diff,
        "std_diff": std_diff,
        "passed_placeholder_check": passed,
    }
    return passed, stats

In [7]:
def search_best_negative_set(
    positives: pd.DataFrame,
    negative_candidates: pd.DataFrame,
    pos_score_col: str,
    neg_score_col: str,
    n_trials: int = 500,
    require_distribution_pass: bool = False,
) -> Tuple[MatchResult, dict]:
    best_match = None
    best_stats = None
    best_key = None

    for seed in range(n_trials):
        result = greedy_match_scores(
            positives=positives,
            negatives=negative_candidates,
            pos_score_col=pos_score_col,
            neg_score_col=neg_score_col,
            random_seed=seed,
            shuffle_positives=True,
        )

        selected = result.selected_negatives.copy()

        if len(selected) != len(positives):
            continue

        passed, stats = run_distribution_check(
            positives=positives,
            negatives=selected,
            pos_score_col=pos_score_col,
            neg_score_col=neg_score_col,
        )

        key = (
            0 if (passed or not require_distribution_pass) else 1,
            result.unmatched_positive_count,
            -result.exact_match_count,
            result.total_absolute_difference,
            result.mean_absolute_difference,
        )

        if best_key is None or key < best_key:
            best_key = key
            best_match = result
            best_stats = stats

        if require_distribution_pass and passed and result.total_absolute_difference == 0:
            break

    if best_match is None or best_stats is None:
        raise RuntimeError("Failed to find a valid matched negative set.")

    return best_match, best_stats

In [26]:
positives_file = "acs_pos_best.csv"
negatives_file = "candidates_acs_neg_best.csv"

pos_score_col = "best_score"
neg_score_col = "best_score"

pos_id_col = "sequence_id"
neg_id_col = "sequence_id"

n_trials = 10
require_distribution_pass = False

use_rounding = True
round_digits = 4

In [27]:
positives = load_scores(positives_file, pos_score_col, pos_id_col)
negative_candidates = load_scores(negatives_file, neg_score_col, neg_id_col)

if use_rounding:
    positives = positives.copy()
    negative_candidates = negative_candidates.copy()
    positives["_match_score"] = positives[pos_score_col].round(round_digits)
    negative_candidates["_match_score"] = negative_candidates[neg_score_col].round(round_digits)
    match_pos_score_col = "_match_score"
    match_neg_score_col = "_match_score"
else:
    match_pos_score_col = pos_score_col
    match_neg_score_col = neg_score_col

print("Positives:", positives.shape)
print("Negative candidates:", negative_candidates.shape)
print(positives.head())
print(negative_candidates.head())

Positives: (298, 3)
Negative candidates: (5920, 3)
  sequence_id  best_score  _match_score
0      seq_10   12.452863       12.4529
1     seq_101   13.359879       13.3599
2     seq_105   13.754595       13.7546
3     seq_106   13.289688       13.2897
4     seq_108   15.225706       15.2257
  sequence_id  best_score  _match_score
0       seq_0   10.569181       10.5692
1       seq_1    8.335455        8.3355
2      seq_10    9.239733        9.2397
3     seq_100   14.800111       14.8001
4    seq_1000    7.152935        7.1529


In [28]:
best_match, best_stats = search_best_negative_set(
    positives=positives,
    negative_candidates=negative_candidates,
    pos_score_col=match_pos_score_col,
    neg_score_col=match_neg_score_col,
    n_trials=n_trials,
    require_distribution_pass=require_distribution_pass,
)

selected_negatives = best_match.selected_negatives.copy()

print("Selected negatives:", len(selected_negatives))
print("Exact matches:", best_match.exact_match_count)
print("Unmatched positives:", best_match.unmatched_positive_count)
print("Total absolute difference:", best_match.total_absolute_difference)
print("Mean absolute difference:", best_match.mean_absolute_difference)

pd.DataFrame([best_stats])

Selected negatives: 298
Exact matches: 44
Unmatched positives: 0
Total absolute difference: 44.29309999999997
Mean absolute difference: 0.14863456375838915


,positive_n,negative_n,positive_mean,negative_mean,positive_median,negative_median,positive_std,negative_std,mean_diff,median_diff,std_diff,passed_placeholder_check
0,298,298,11.706804,11.597003,11.8879,11.88905,2.394567,2.164111,0.109801,0.00115,0.230456,False


In [29]:
selected_negatives.head()

,sequence_id,best_score,_match_score,_neg_idx,_matched_positive_score,_abs_diff
0,seq_5232,11.028478,11.0285,4705,11.0305,0.0020
1,seq_4229,10.669367,10.6694,3590,10.6692,0.0002
2,seq_5431,10.387516,10.3875,4926,10.3875,0.0000
3,seq_1404,13.719253,13.7193,452,13.7214,0.0021
4,seq_5371,12.275453,12.2755,4859,12.2731,0.0024


In [30]:
output_file = "selected_acs_negatives.csv"
stats_file = "selected_acs_negatives.stats.csv"

selected_negatives.to_csv(output_file, index=False)

stats_row = {
    "selected_n": len(selected_negatives),
    "positive_n": len(positives),
    "exact_match_count": best_match.exact_match_count,
    "unmatched_positive_count": best_match.unmatched_positive_count,
    "total_absolute_difference": best_match.total_absolute_difference,
    "mean_absolute_difference": best_match.mean_absolute_difference,
    **best_stats,
}

pd.DataFrame([stats_row]).to_csv(stats_file, index=False)

print("Saved:", output_file)
print("Saved:", stats_file)
pd.DataFrame([stats_row])

Saved: selected_acs_negatives.csv
Saved: selected_acs_negatives.stats.csv


,selected_n,positive_n,exact_match_count,unmatched_positive_count,total_absolute_difference,mean_absolute_difference,negative_n,positive_mean,negative_mean,positive_median,negative_median,positive_std,negative_std,mean_diff,median_diff,std_diff,passed_placeholder_check
0,298,298,44,0,44.2931,0.148635,298,11.706804,11.597003,11.8879,11.88905,2.394567,2.164111,0.109801,0.00115,0.230456,False


compare the mean motif score of the negative and positives and if check if no significant statistical difference be there

In [ ]:
import pandas as pd
from scipy import stats

pos = pd.read_csv("acs_pos_best.csv")
neg = pd.read_csv("selected_acs_negatives.csv")

print("=== BEST SCORE COMPARISON ===")
u_score = stats.mannwhitneyu(pos["best_score"], neg["best_score"])
print("Mann-Whitney p-value:", u_score.pvalue)

ks_score = stats.ks_2samp(pos["best_score"], neg["best_score"])
print("KS test p-value:", ks_score.pvalue)

# print("\n=== BEST P-VALUE COMPARISON ===")
# u_p = stats.mannwhitneyu(pos["best_pvalue"], neg["best_pvalue"])
# print("Mann-Whitney p-value:", u_p.pvalue)

# ks_p = stats.ks_2samp(pos["best_pvalue"], neg["best_pvalue"])
# print("KS test p-value:", ks_p.pvalue)


=== BEST SCORE COMPARISON ===
Mann-Whitney p-value: 0.7084389617855753
KS test p-value: 0.5806657981612027
